In [1]:
#%pip install cassandra-driver 

In [2]:
from pyspark.sql import SparkSession 
from pyspark.sql import functions as F
from cassandra.cluster import Cluster

## Partie 1 : Connecter spark à Cassandra

In [3]:
spark = (SparkSession.builder 
    .appName("TP-Cassandra-Spark") 
    .master("local[4]") 
    .config("spark.sql.shuffle.partitions", "4") 
    .getOrCreate()) 
 
sc = spark.sparkContext 
print("Version Spark :", spark.version) 
print("Master        :", sc.master) 
print("Parallélisme  :", sc.defaultParallelism) 

Version Spark : 4.2.0
Master        : local[4]
Parallélisme  : 4


In [4]:
cluster = Cluster(["cass1"]) 
session = cluster.connect() 
print("Connexion Cassandra réussie")

Connexion Cassandra réussie


In [5]:
session.set_keyspace("velib_cluster")
rows = session.execute("""
SELECT * FROM stations_velib;
""")

# Dépile les row, ne pas laisser décommenter sinon le DF créé après est incomplet
#for row in rows :
#    print(row)

## Partie 2 : Créer votre DataFrame Spark

In [6]:
columns = rows.column_names
data = [tuple(row) for row in rows]

df = spark.createDataFrame(data, columns)

df.show(5)
df.printSchema()
df.count()

+----------+------------------+--------+--------------------+--------------------+--------------------+
|station_id|bornes_disponibles|capacite|derniere_mise_a_jour|         nom_station|vatiques_disponibles|
+----------+------------------+--------+--------------------+--------------------+--------------------+
|      3004|                 9|      18| 2026-10-07 08:27:33|Place de la Répub...|                   9|
|     15061|                 4|      22| 2026-10-07 07:28:39|  Square Desnouettes|                  17|
|     15047|                50|      52| 2026-10-07 08:26:46| Morillons - Dantzig|                   2|
|     23202|                26|      28| 2026-10-07 07:21:22|Henri Barbusse - ...|                   2|
|      9013|                 2|      24| 2026-10-07 07:28:29|      Mairie du 9ème|                  21|
+----------+------------------+--------+--------------------+--------------------+--------------------+
only showing top 5 rows
root
 |-- station_id: string (nullable =

1240

On utilise les données de l'API velib de Paris récupérées dans Cassandra au TP précedent.

Colonnes principales de l'analyse : `vatiques_disponibles`, `capacite`, `derniere_mise_a_jour` et `nom_station` (si on avait plusieurs tables on favoriserait station_id pour les jointures)

Information souhaité : Détails de l'occupation/l'activité des station sur une plage horaire

## Partie 3 : Transformations Spark

### Sélection

In [7]:
df_clean = df.select(["nom_station", "vatiques_disponibles","capacite","derniere_mise_a_jour"])
df_clean.show(5)

+--------------------+--------------------+--------+--------------------+
|         nom_station|vatiques_disponibles|capacite|derniere_mise_a_jour|
+--------------------+--------------------+--------+--------------------+
|Place de la Répub...|                   9|      18| 2026-10-07 08:27:33|
|  Square Desnouettes|                  17|      22| 2026-10-07 07:28:39|
| Morillons - Dantzig|                   2|      52| 2026-10-07 08:26:46|
|Henri Barbusse - ...|                   2|      28| 2026-10-07 07:21:22|
|      Mairie du 9ème|                  21|      24| 2026-10-07 07:28:29|
+--------------------+--------------------+--------+--------------------+
only showing top 5 rows


### Filtres

In [8]:
print("=================\nMoins de 5 vélos\n=================")
df_clean.filter(F.col("vatiques_disponibles") < 5).show(5)
print("=================\nPlus de 20 vélos\n=================")
df_clean.filter(F.col("vatiques_disponibles") > 20).show(5)

Moins de 5 vélos
+--------------------+--------------------+--------+--------------------+
|         nom_station|vatiques_disponibles|capacite|derniere_mise_a_jour|
+--------------------+--------------------+--------+--------------------+
| Morillons - Dantzig|                   2|      52| 2026-10-07 08:26:46|
|Henri Barbusse - ...|                   2|      28| 2026-10-07 07:21:22|
|Louis Ganne - Davout|                   1|      30| 2026-10-07 07:27:39|
|Liard - Amiral Mo...|                   1|      28| 2026-10-07 08:28:48|
|Place Adolphe Che...|                   3|      63| 2026-10-07 08:27:47|
+--------------------+--------------------+--------+--------------------+
only showing top 5 rows
Plus de 20 vélos
+--------------------+--------------------+--------+--------------------+
|         nom_station|vatiques_disponibles|capacite|derniere_mise_a_jour|
+--------------------+--------------------+--------+--------------------+
|      Mairie du 9ème|                  21|      24| 2

### Colonne calculée

In [9]:
df_clean = df_clean.withColumn(
    "pct_remplissage",
    F.when(
        F.col("vatiques_disponibles") > 0,
        F.round((F.col("vatiques_disponibles") / F.col("capacite")) * 100, 2)
    ).otherwise(None)
)
df_clean = df_clean.withColumn(
    "etat_remplissage",
    F.when(
        (F.col("pct_remplissage").isNotNull()) & (F.col("pct_remplissage") < 25),
        "VIDE"
    ).when(
        (F.col("pct_remplissage").isNotNull()) & (F.col("pct_remplissage") <= 75),
        "OK"
    ).when(
        (F.col("pct_remplissage").isNotNull()) & (F.col("pct_remplissage") > 75),
        "PLEIN"
    ).otherwise("INCONNU")
)
df_clean.show(5)

+--------------------+--------------------+--------+--------------------+---------------+----------------+
|         nom_station|vatiques_disponibles|capacite|derniere_mise_a_jour|pct_remplissage|etat_remplissage|
+--------------------+--------------------+--------+--------------------+---------------+----------------+
|Place de la Répub...|                   9|      18| 2026-10-07 08:27:33|           50.0|              OK|
|  Square Desnouettes|                  17|      22| 2026-10-07 07:28:39|          77.27|           PLEIN|
| Morillons - Dantzig|                   2|      52| 2026-10-07 08:26:46|           3.85|            VIDE|
|Henri Barbusse - ...|                   2|      28| 2026-10-07 07:21:22|           7.14|            VIDE|
|      Mairie du 9ème|                  21|      24| 2026-10-07 07:28:29|           87.5|           PLEIN|
+--------------------+--------------------+--------+--------------------+---------------+----------------+
only showing top 5 rows


## Partie 4 : Actions et agrégations

In [10]:
df_clean.groupBy("etat_remplissage").agg(
    F.count("*").alias("nb_stations"),
    F.mean("pct_remplissage").alias("moy_pct_remplissage"),
    F.min("pct_remplissage").alias("min_pct_remplissage"),
    F.max("pct_remplissage").alias("max_pct_remplissage"),
).show(5)

+----------------+-----------+-------------------+-------------------+-------------------+
|etat_remplissage|nb_stations|moy_pct_remplissage|min_pct_remplissage|max_pct_remplissage|
+----------------+-----------+-------------------+-------------------+-------------------+
|              OK|        436| 48.702660550458724|               25.0|               75.0|
|           PLEIN|        241|  90.12883817427384|              75.56|             181.82|
|            VIDE|        450|  11.21857777777778|               1.82|              24.49|
|         INCONNU|        113|               NULL|               NULL|               NULL|
+----------------+-----------+-------------------+-------------------+-------------------+



## Partie 5 : Partitions et Lazy Evaluation

In [12]:
print(df_clean.rdd.getNumPartitions())

sc.setJobGroup("demo_lazy","")

def nb_jobs():
    return len(sc.statusTracker().getJobIdsForGroup("demo_lazy"))

print("Jobs avant transformations :", nb_jobs())

df_transfo = (
    df_clean.repartition(1)
    .withColumn("pct_1dec", F.round("pct_remplissage", 1))
    .withColumn("capacite_x2", F.col("capacite") * 2)
    .filter(F.col("capacite") >= 20)
)
print("Jobs après transformations :", nb_jobs())

df_transfo.show(5)
print("Jobs après show() :", nb_jobs())    
print("Partitions finales :", df_transfo.rdd.getNumPartitions())

4
Jobs avant transformations : 0
Jobs après transformations : 0
+--------------------+--------------------+--------+--------------------+---------------+----------------+--------+-----------+
|         nom_station|vatiques_disponibles|capacite|derniere_mise_a_jour|pct_remplissage|etat_remplissage|pct_1dec|capacite_x2|
+--------------------+--------------------+--------+--------------------+---------------+----------------+--------+-----------+
|  Square Desnouettes|                  17|      22| 2026-10-07 07:28:39|          77.27|           PLEIN|    77.3|         44|
| Morillons - Dantzig|                   2|      52| 2026-10-07 08:26:46|           3.85|            VIDE|     3.9|        104|
|Henri Barbusse - ...|                   2|      28| 2026-10-07 07:21:22|           7.14|            VIDE|     7.1|         56|
|      Mairie du 9ème|                  21|      24| 2026-10-07 07:28:29|           87.5|           PLEIN|    87.5|         48|
|Gare de Lyon - Pl...|                  

**Partitions** : 4 au départ → `repartition(1)` → 1 partition.

**Jobs** : 0 avant les transformations, 0 après, 2 après `show()` — les transformations seules ne lancent rien.

**Pourquoi la Lazy Evaluation ?**

Spark définit un plan avant de faire ses calculs, la Lazy evalutation lui permet d'optimiser ce plan et ainsi d'économiser des ressources. Son plan est ainsi un plan global et non centré sur 1 seule opération.

## Partie 6 : Observer l'exécution avec Spark UI

Observations depuis `localhost:4040` :

- **18 Jobs**, tous `SUCCEEDED` — 1 à 2 Jobs par action
- **24 Stages** : 18 `COMPLETE`, 6 `SKIPPED` (résultats déjà en cache)
- **39 Tasks exécutées**, **1 ou 4 par Stage** (`local[4]` = 4 partitions)
- Durées : 0,03 s à 2,5 s, moyenne 0,35 s
- **Shuffle** : `repartition(1)` → 4 Tasks écrivent 236 B / 4 enregistrements, **1 seule Task** les relit ; les agrégations vont jusqu'à 58 177 B / 1 118 enregistrements (`Exchange` visible dans le plan d'exécution)
- Le Stage final de `show()` n'a qu'**1 Task** : Spark agrège le résultat sur une seule partition avant d'afficher

## Partie 7 : Votre analyse métier

**Question métier :** quelles stations présentent un risque de rupture de service, et à quelle heure ce risque est-il le plus élevé ?

```text
Lecture Cassandra → DataFrame → Sélection → Filtre → Colonne calculée → Agrégation → Action
```

In [13]:
rows_full = session.execute("""
SELECT station_id, nom_station, vatiques_disponibles, capacite, derniere_mise_a_jour
FROM stations_velib;
""")

columns_full = rows_full.column_names
data_full = [tuple(row) for row in rows_full]

df_src = spark.createDataFrame(data_full, columns_full)
df_src.count()

1240

In [14]:
analyse = (
    df_src.select("nom_station", "vatiques_disponibles", "capacite", "derniere_mise_a_jour")
    .filter((F.col("capacite") > 0) & F.col("vatiques_disponibles").isNotNull())
    .withColumn("pct_dispo", F.round(F.col("vatiques_disponibles") / F.col("capacite") * 100, 2))
    .withColumn("heure", F.hour("derniere_mise_a_jour"))
    .withColumn(
        "niveau_risque",
        F.when(F.col("pct_dispo") < 25, "RUPTURE")
         .when(F.col("pct_dispo") <= 75, "STABLE")
         .otherwise("SURSTOCK")
    )
)

In [15]:
par_heure = (
    analyse.groupBy("heure")
    .agg(
        F.count("*").alias("nb_stations"),
        F.round(F.mean("pct_dispo"), 2).alias("moy_pct_dispo"),
        F.sum(F.when(F.col("niveau_risque") == "RUPTURE", 1).otherwise(0)).alias("nb_rupture"),
        F.round(
            F.sum(F.when(F.col("niveau_risque") == "RUPTURE", 1).otherwise(0))
            / F.count("*") * 100, 2
        ).alias("pct_rupture"),
    )
    .orderBy("heure")
)

par_heure.show(truncate=False)
analyse.filter(F.col("niveau_risque") == "RUPTURE").count()

+-----+-----------+-------------+----------+-----------+
|heure|nb_stations|moy_pct_dispo|nb_rupture|pct_rupture|
+-----+-----------+-------------+----------+-----------+
|0    |1          |0.0          |1         |100.0      |
|2    |3          |18.22        |2         |66.67      |
|5    |1          |0.0          |1         |100.0      |
|7    |689        |39.06        |311       |45.14      |
|8    |536        |39.09        |239       |44.59      |
|9    |2          |0.0          |2         |100.0      |
|11   |1          |0.0          |1         |100.0      |
|15   |1          |0.0          |1         |100.0      |
|16   |1          |3.0          |1         |100.0      |
|17   |1          |13.64        |1         |100.0      |
|18   |1          |70.0         |0         |0.0        |
|19   |1          |0.0          |1         |100.0      |
+-----+-----------+-------------+----------+-----------+



561

- **Données** : l'état des stations Vélib à chaque mise à jour, pour mesurer l'offre de vélos.
- **Transformations** : `pct_dispo` permet de comparer des stations de capacités différentes ; `niveau_risque` traduit ce taux en catégories métier.
- **Agrégation** : regrouper par heure donne directement les plages où lancer des opérations de rééquilibrage.
- **Exécution** : les transformations ne construisent qu'un plan, `show()` et `count()` déclenchent les Jobs observés dans la Spark UI (Shuffle visible sur l'`groupBy` et l'`orderBy`).